# step07_synthesis — Synthesize Findings and Answer Research Questions

Bring the previous outputs together and state what the metadata can and cannot support.

Run cells from top to bottom. Shared helpers and file paths are in `analysis_common.py`. Outputs go to `analysis_results/` and `figures/`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'analysis' / 'analysis_common.py').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'analysis'))
from analysis_common import *
df=load_revisions()
summary=pd.read_csv(OUTPUT_DIR/'step01_dataset_summary.csv').iloc[0]
growth=pd.read_csv(OUTPUT_DIR/'step03_article_growth.csv')
editors=pd.read_csv(OUTPUT_DIR/'step04_article_editor_metrics.csv')
categories=pd.read_csv(OUTPUT_DIR/'step06_category_summary.csv')
spikes=pd.read_csv(OUTPUT_DIR/'step02_article_monthly_spikes.csv')
valid_spikes=spikes[spikes.is_spike].sort_values('revisions',ascending=False).head(10)
findings=[
('Dataset scale',f"{int(summary.revision_records):,} revision records cover {int(summary.articles)} articles; median {summary.median_revisions_per_article:,.0f} records per article."),
('Time coverage',f"Valid timestamps span {summary.earliest_timestamp_utc} through {summary.latest_timestamp_utc} ({summary.coverage_days/365.2425:.1f} years)."),
('Article growth',f"The largest net observed byte increase is {growth.iloc[0].article_title} (+{growth.iloc[0].net_change_bytes:,.0f} bytes); byte size is a proxy, not a measure of historical completeness."),
('Contributor concentration',f"Median top-10 username revision share across articles is {editors.top10_revision_share_pct.median():.1f}%. Usernames and metadata do not establish identity or expertise."),
('Event-category comparison',f"Category medians describe the articles assigned to each primary category; category sample sizes range from {int(categories.articles.min())} to {int(categories.articles.max())}."),
('Editing spikes','Per-article robust monthly outliers are listed in step02_article_monthly_spikes.csv. A spike is an observed anomaly, not evidence of its cause.'),
('Content changes','The retrieved sample shows a vandalism/reversion cycle, maintenance-template edits, a bibliography addition, and a casualty-list removal; the small sample cannot characterize all revisions or verify historical claims.')
]
for label,text in findings: print(f'\n{label}: {text}')


In [ ]:
questions=pd.DataFrame([{'research_question':'How does coverage of historical events evolve over time?','answer':'Describe observed revision timelines and article byte-size trajectories; the collected records may be incomplete or sampled.','evidence':'step02_monthly_activity_by_article.csv; step03_article_growth.csv','confidence':'Descriptive'}, {'research_question':'When do substantial changes occur?','answer':'Use per-article monthly robust outliers and large observed byte changes; these are candidate events.','evidence':'step02_article_monthly_spikes.csv; step03_largest_size_changes.csv','confidence':'Descriptive'}, {'research_question':'Do spikes coincide with public attention or related events?','answer':'Not established by revision metadata. Requires independently dated external sources for each spike.','evidence':'step02_spike_context_review.csv','confidence':'Unverified'}, {'research_question':'What kinds of changes are made to historical information?','answer':'The selected diff sample documents vandalism/reversion, maintenance templates, references, and a list-item removal; it is too small to characterize all content changes.','evidence':'step05_revision_case_studies.csv','confidence':'Depends on retrieval and manual review'}, {'research_question':'Do patterns differ between event types?','answer':'Compare category medians and distributions while accounting for category/article counts.','evidence':'step06_category_summary.csv','confidence':'Descriptive; mapping-sensitive'}])
save_table(questions,'step07_research_question_answers.csv'); display(questions)
(OUTPUT_DIR/'step07_findings_summary.md').write_text('# Findings summary\n\n'+'\n'.join(f'## {label}\n\n{text}\n' for label,text in findings)+'\n## Limitations\n\nRevision metadata does not encode the complete article text, reliably explain editing motivation, or establish contributor identity. Contextual claims need independent dated sources. Categories are researcher-assigned.\n',encoding='utf-8')


## Chart selection for the report

Recommended core visuals: annual revision activity (Step 2), net article growth and selected size timelines (Step 3), contributor concentration and account-type proxy (Step 4), verified before/after diffs only where retrieval succeeded (Step 5), and category medians with sample counts (Step 6). Exclude any chart that does not answer a research question.

In [ ]:
for index,(label,text) in enumerate(findings):
    if label == 'Content changes':
        findings[index] = (label,'The retrieved sample documents vandalism/reversion, maintenance-template edits, a bibliography addition, and a list-item removal; it is too small to characterize all content changes or verify historical claims.')
questions.loc[questions.research_question.eq('What kinds of changes are made to historical information?'),'answer']='The sample diff review documents vandalism/reversion, maintenance templates, a bibliography addition, and a casualty-list removal; it cannot characterize all changes or establish the correctness of claims.'
questions.loc[questions.research_question.eq('What kinds of changes are made to historical information?'),'confidence']='Observed in selected diffs; limited sample'
save_table(questions,'step07_research_question_answers.csv')
(OUTPUT_DIR/'step07_findings_summary.md').write_text('# Findings summary\n\n'+'\n'.join(f'## {label}\n\n{text}\n' for label,text in findings)+'\n## Limitations\n\nRevision metadata does not encode all article text, explain editing motivation, or establish contributor identity. Contextual claims need independent dated sources. Categories are researcher-assigned.\n',encoding='utf-8')
display(questions)
